# Trabajo Final Visión por Computadora II - CEIA - UBA
## Grad-CAM: Visualización de Errores - DenseNet121

**Resumen**

Este notebook implementa **Grad-CAM** (Gradient-weighted Class Activation Mapping) para visualizar las regiones de las imágenes donde el modelo **DenseNet121** se enfoca al realizar predicciones, especialmente cuando comete errores de clasificación.

**Grad-CAM** es una técnica de interpretabilidad que:
- Identifica las regiones de la imagen que más influyen en la decisión del modelo
- Permite validar si el modelo se enfoca en regiones anatómicamente relevantes
- Ayuda a detectar posibles sesgos o características espurias
- Facilita el análisis de errores para mejorar el modelo

Este análisis es especialmente relevante en el contexto médico, donde es crucial entender qué regiones de las imágenes de tomografía computada utiliza el modelo para tomar decisiones.


## Importar librerías


In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
from pathlib import Path
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models


## Configuración y carga de datos


In [ ]:
# Configuración de rutas
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    data_dir = Path('/content/drive/MyDrive/Data_Clean')
else:
    data_dir = Path('./Data_Clean')

# Parámetros de imágenes (deben coincidir con el entrenamiento)
ANCHO_IMAGENES = 224
ALTO_IMAGENES = 224
CANTIDAD_CLASES = 4

# Configuración del dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

# Verificar que existe el directorio
if not data_dir.exists():
    raise FileNotFoundError(f"No se encontró el directorio {data_dir}")
    
print(f"Directorio de datos: {data_dir}")


In [ ]:
# Transformaciones para test (sin data augmentation, solo normalización)
test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # 3 canales porque DenseNet espera RGB
    transforms.Resize(size=(ANCHO_IMAGENES, ALTO_IMAGENES)),
    transforms.CenterCrop(size=(ANCHO_IMAGENES, ALTO_IMAGENES)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Cargar dataset de test
test_dataset = datasets.ImageFolder(root=data_dir / "test", transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Obtener nombres de clases y mapeo
class_name_mapping = {
    'adenocarcinoma_left': 'Adenocarcinoma',
    'large.cell.carcinoma_left': 'Large Cell Carcinoma',
    'normal': 'Normal',
    'squamous.cell.carcinoma_left': 'Squamous Cell Carcinoma'
}

class_names = [class_name_mapping.get(name, name) for name in test_dataset.classes]

print(f"Clases: {class_names}")
print(f"Imágenes de test: {len(test_dataset)}")


## Cargar modelo DenseNet121 entrenado


In [ ]:
# Cargar arquitectura DenseNet121 con pesos de ImageNet
from torchvision.models import DenseNet121_Weights

weights = DenseNet121_Weights.IMAGENET1K_V1
densenet121_model = models.densenet121(weights=weights)

# Reemplazar la última capa para nuestro número de clases
in_features = densenet121_model.classifier.in_features
densenet121_model.classifier = nn.Linear(in_features, CANTIDAD_CLASES)

# Cargar pesos entrenados
model_path = data_dir / "DenseNet121_best.pth"
if model_path.exists():
    densenet121_model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Modelo cargado desde: {model_path}")
else:
    raise FileNotFoundError(f"No se encontró el modelo en {model_path}")

# Mover modelo al dispositivo y poner en modo evaluación
densenet121_model.to(device)
densenet121_model.eval()

print("Modelo DenseNet121 cargado y listo para Grad-CAM")


## Implementación de Grad-CAM

### Selección de la capa objetivo

Grad-CAM extrae las activaciones de una capa convolucional específica. Para DenseNet121, utilizamos **`features.norm5`**, que es la última capa de normalización antes del pooling global y la capa fully-connected.

**¿Por qué norm5?**
- **Características de alto nivel**: `norm5` contiene las características más semánticas y abstractas aprendidas por la red después de pasar por todos los bloques densos
- **Última capa antes del pooling**: Es la última capa que mantiene información espacial antes del pooling global y la clasificación
- **Mejor interpretabilidad**: Las activaciones en esta capa están más directamente relacionadas con las decisiones de clasificación
- **Estándar en la literatura**: Es la capa más comúnmente usada para Grad-CAM en arquitecturas DenseNet

**Estructura de DenseNet121:**
```
Input (224x224x3)
  ↓
conv0, norm0, relu0, pool0
  ↓
denseblock1, transition1 → características de bajo nivel
  ↓
denseblock2, transition2 → características de nivel medio
  ↓
denseblock3, transition3 → características de nivel medio-alto
  ↓
denseblock4 → características de ALTO NIVEL (objetos, patrones complejos)
  ↓
norm5 → última normalización antes del pooling ⭐
  ↓
avgpool (pooling global)
  ↓
classifier (clasificador)
```

**Alternativas:**
- `features.denseblock4`: Proporcionaría mapas más detallados pero menos semánticos
- Capas anteriores: Demasiado detalladas, difícil de interpretar
- Capas después de `avgpool`: No tienen información espacial


In [ ]:
class GradCAM:
    """
    Implementación de Grad-CAM para DenseNet121.
    Captura las activaciones de la última capa antes del pooling (norm5)
    y calcula los mapas de calor basados en los gradientes.
    """
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Registrar hooks para capturar activaciones y gradientes
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_backward_hook(self.save_gradient)
    
    def save_activation(self, module, input, output):
        """Guarda las activaciones de la capa objetivo"""
        self.activations = output.detach()
    
    def save_gradient(self, module, grad_input, grad_output):
        """Guarda los gradientes de la capa objetivo"""
        # grad_output es una tupla, tomamos el primer elemento
        if grad_output[0] is not None:
            self.gradients = grad_output[0].detach()
    
    def generate_cam(self, input_image, class_idx=None):
        """
        Genera el mapa de activación de clase (CAM) usando Grad-CAM
        
        Args:
            input_image: Tensor de imagen de entrada (1, C, H, W)
            class_idx: Índice de la clase. Si es None, usa la clase predicha
        
        Returns:
            cam: Mapa de calor normalizado (H, W)
        """
        self.model.eval()
        
        # Asegurar que la imagen requiere gradientes
        input_image = input_image.requires_grad_(True)
        
        # Forward pass
        output = self.model(input_image)
        
        if class_idx is None:
            class_idx = output.argmax(dim=1)
        
        # Backward pass
        self.model.zero_grad()
        output[0, class_idx].backward(retain_graph=True)
        
        # Calcular pesos de los canales usando gradientes
        gradients = self.gradients[0]  # (C, H, W)
        activations = self.activations[0]  # (C, H, W)
        
        # Mover a CPU si es necesario para evitar problemas de memoria
        if gradients.is_cuda:
            gradients = gradients.cpu()
        if activations.is_cuda:
            activations = activations.cpu()
        
        # Promedio global de los gradientes para cada canal
        weights = torch.mean(gradients, dim=(1, 2))  # (C,)
        
        # Ponderar las activaciones por los pesos
        cam = torch.zeros(activations.shape[1:], dtype=torch.float32)
        for i, w in enumerate(weights):
            cam += w * activations[i, :, :]
        
        # Aplicar ReLU (solo valores positivos)
        cam = torch.relu(cam)
        
        # Normalizar entre 0 y 1
        cam = cam - cam.min()
        cam = cam / cam.max() if cam.max() > 0 else cam
        
        return cam.cpu().numpy()
    
    def overlay_heatmap(self, original_image, cam, alpha=0.4):
        """
        Superpone el mapa de calor sobre la imagen original
        
        Args:
            original_image: Imagen original (H, W, 3) en formato numpy
            cam: Mapa de calor (H, W)
            alpha: Transparencia del mapa de calor
        
        Returns:
            overlaid: Imagen con mapa de calor superpuesto
            heatmap: Mapa de calor en color
        """
        # Redimensionar CAM a tamaño de la imagen original
        cam_resized = cv2.resize(cam, (original_image.shape[1], original_image.shape[0]))
        cam_resized = np.uint8(255 * cam_resized)
        
        # Aplicar colormap (jet es común para mapas de calor)
        heatmap = cv2.applyColorMap(cam_resized, cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        
        # Superponer
        overlaid = (alpha * heatmap + (1 - alpha) * original_image).astype(np.uint8)
        
        return overlaid, heatmap

# Crear instancia de GradCAM apuntando a norm5 (última capa antes del pooling)
gradcam = GradCAM(densenet121_model, densenet121_model.features.norm5)
print("GradCAM inicializado para features.norm5")


In [ ]:
def denormalize_image(tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    """
    Desnormaliza una imagen tensor para visualización
    """
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    tensor = tensor * std + mean
    tensor = torch.clamp(tensor, 0, 1)
    return tensor


## Identificar errores de clasificación


In [ ]:
# Identificar imágenes con errores y ejemplos correctos por clase
densenet121_model.eval()
errors = []  # Lista de errores con información completa
correct_examples = {i: [] for i in range(CANTIDAD_CLASES)}  # Ejemplos correctos por clase

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(test_loader):
        images_gpu = images.to(device)
        labels_gpu = labels.to(device)
        
        outputs = densenet121_model(images_gpu)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        
        # Identificar errores y aciertos
        for i in range(len(labels)):
            label = labels[i].item()
            pred = preds[i].item()
            
            if pred != label:
                # Es un error
                errors.append({
                    'image': images[i],
                    'label': label,
                    'pred': pred,
                    'probs': probs[i].cpu().numpy(),
                    'batch_idx': batch_idx,
                    'item_idx': i
                })
            else:
                # Es un acierto - guardar hasta 3 por clase
                if len(correct_examples[label]) < 3:
                    correct_examples[label].append({
                        'image': images[i],
                        'label': label,
                        'pred': pred,
                        'probs': probs[i].cpu().numpy(),
                        'batch_idx': batch_idx,
                        'item_idx': i
                    })
        
        # Continuar hasta tener suficientes errores y ejemplos correctos
        if len(errors) >= 10:
            # Verificar que tenemos al menos 3 ejemplos correctos para cada clase que tiene errores
            error_classes = set([e['label'] for e in errors])
            all_have_examples = all(len(correct_examples[cls]) >= 3 for cls in error_classes)
            if all_have_examples:
                break

print(f"Se encontraron {len(errors)} errores en las primeras imágenes del test set")
print(f"\nDistribución de errores:")
error_counts = {}
for error in errors:
    true_class = class_names[error['label']]
    pred_class = class_names[error['pred']]
    key = f"{true_class} -> {pred_class}"
    error_counts[key] = error_counts.get(key, 0) + 1

for error_type, count in sorted(error_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {error_type}: {count}")

print(f"\nEjemplos correctos encontrados por clase:")
for cls_idx, cls_name in enumerate(class_names):
    print(f"  {cls_name}: {len(correct_examples[cls_idx])} ejemplos")


In [ ]:
# Comparar errores con aciertos de la misma clase
num_errors_to_show = min(5, len(errors))

if num_errors_to_show > 0:
    for error_idx, error in enumerate(errors[:num_errors_to_show]):
        true_label = error['label']
        pred_label = error['pred']
        
        # Verificar que tenemos ejemplos correctos de esta clase
        if len(correct_examples[true_label]) < 3:
            print(f"Advertencia: No hay suficientes ejemplos correctos para la clase {class_names[true_label]}")
            continue
        
        # Crear figura: 1 fila para el error + 3 filas para aciertos (4 columnas cada una)
        fig, axes = plt.subplots(4, 4, figsize=(28, 28))
        
        # ===== FILA 1: ERROR =====
        error_image_tensor = error['image'].unsqueeze(0).to(device)
        error_img_denorm = denormalize_image(error['image'])
        error_img_np = (error_img_denorm.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        
        # CAM para clase predicha (incorrecta)
        error_cam_pred = gradcam.generate_cam(error_image_tensor, class_idx=pred_label)
        # CAM para clase real
        error_cam_true = gradcam.generate_cam(error_image_tensor, class_idx=true_label)
        
        # Columna 1: Imagen original del error
        axes[0, 0].imshow(error_img_np, cmap='gray')
        axes[0, 0].set_title(f'ERROR\nTrue: {class_names[true_label]}\nPred: {class_names[pred_label]}', 
                             fontsize=14, color='red', fontweight='bold')
        axes[0, 0].axis('off')
        
        # Columna 2: Grad-CAM para clase predicha (incorrecta)
        axes[0, 1].imshow(error_img_np, cmap='gray')
        axes[0, 1].imshow(error_cam_pred, alpha=0.5, cmap='jet')
        axes[0, 1].set_title(f'Grad-CAM: Predicha\n({class_names[pred_label]})\nConf: {error["probs"][pred_label]:.2%}', 
                            fontsize=13, fontweight='bold')
        axes[0, 1].axis('off')
        
        # Columna 3: Grad-CAM para clase real
        axes[0, 2].imshow(error_img_np, cmap='gray')
        axes[0, 2].imshow(error_cam_true, alpha=0.5, cmap='jet')
        axes[0, 2].set_title(f'Grad-CAM: Real\n({class_names[true_label]})\nConf: {error["probs"][true_label]:.2%}', 
                            fontsize=13, fontweight='bold')
        axes[0, 2].axis('off')
        
        # Columna 4: Heatmap superpuesto
        overlaid_error, _ = gradcam.overlay_heatmap(error_img_np, error_cam_true)
        axes[0, 3].imshow(overlaid_error)
        axes[0, 3].set_title('Heatmap Superpuesto\n(Clase Real)', fontsize=13)
        axes[0, 3].axis('off')
        
        # ===== FILAS 2-4: 3 EJEMPLOS CORRECTOS =====
        for correct_idx, correct_example in enumerate(correct_examples[true_label][:3]):
            row = correct_idx + 1
            
            correct_image_tensor = correct_example['image'].unsqueeze(0).to(device)
            correct_img_denorm = denormalize_image(correct_example['image'])
            correct_img_np = (correct_img_denorm.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
            
            # CAM para la clase correcta
            correct_cam = gradcam.generate_cam(correct_image_tensor, class_idx=true_label)
            
            # Columna 1: Imagen original
            axes[row, 0].imshow(correct_img_np, cmap='gray')
            axes[row, 0].set_title(f'ACIERTO #{correct_idx+1}\nTrue: {class_names[true_label]}\nPred: {class_names[true_label]}', 
                                   fontsize=14, color='green', fontweight='bold')
            axes[row, 0].axis('off')
            
            # Columna 2: Grad-CAM
            axes[row, 1].imshow(correct_img_np, cmap='gray')
            axes[row, 1].imshow(correct_cam, alpha=0.5, cmap='jet')
            axes[row, 1].set_title(f'Grad-CAM\n({class_names[true_label]})\nConf: {correct_example["probs"][true_label]:.2%}', 
                                  fontsize=13, fontweight='bold')
            axes[row, 1].axis('off')
            
            # Columna 3: Comparación de activación promedio
            # Calcular estadísticas de activación
            error_mean_activation = error_cam_true.mean()
            correct_mean_activation = correct_cam.mean()
            
            axes[row, 2].text(0.5, 0.5, 
                             f'Act. Promedio:\nError: {error_mean_activation:.3f}\nAcierto: {correct_mean_activation:.3f}\n'
                             f'Diferencia: {abs(error_mean_activation - correct_mean_activation):.3f}',
                             ha='center', va='center', fontsize=12,
                             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
            axes[row, 2].axis('off')
            
            # Columna 4: Heatmap superpuesto
            overlaid_correct, _ = gradcam.overlay_heatmap(correct_img_np, correct_cam)
            axes[row, 3].imshow(overlaid_correct)
            axes[row, 3].set_title('Heatmap Superpuesto', fontsize=13)
            axes[row, 3].axis('off')
        
        plt.suptitle(f'Comparación Error vs. Aciertos: {class_names[true_label]} clasificado como {class_names[pred_label]}', 
                     fontsize=18, fontweight='bold', y=0.998)
        plt.tight_layout(rect=[0, 0, 1, 0.995], h_pad=2.0, w_pad=2.0)
        plt.show()
        
        # Análisis estadístico
        error_cams = [error_cam_true]
        correct_cams = [gradcam.generate_cam(correct_example['image'].unsqueeze(0).to(device), class_idx=true_label) 
                       for correct_example in correct_examples[true_label][:3]]
        
        error_max_activations = [cam.max() for cam in error_cams]
        correct_max_activations = [cam.max() for cam in correct_cams]
        error_mean_activations = [cam.mean() for cam in error_cams]
        correct_mean_activations = [cam.mean() for cam in correct_cams]
        
        print(f"\n{'='*60}")
        print(f"Análisis estadístico - Error #{error_idx+1}: {class_names[true_label]} → {class_names[pred_label]}")
        print(f"{'='*60}")
        print(f"Activación máxima:")
        print(f"  Error: {np.mean(error_max_activations):.3f} ± {np.std(error_max_activations):.3f}")
        print(f"  Aciertos: {np.mean(correct_max_activations):.3f} ± {np.std(correct_max_activations):.3f}")
        print(f"\nActivación promedio:")
        print(f"  Error: {np.mean(error_mean_activations):.3f} ± {np.std(error_mean_activations):.3f}")
        print(f"  Aciertos: {np.mean(correct_mean_activations):.3f} ± {np.std(correct_mean_activations):.3f}")
        print(f"\nProbabilidades:")
        print(f"  Error - {class_names[true_label]}: {error['probs'][true_label]:.2%}")
        print(f"  Error - {class_names[pred_label]}: {error['probs'][pred_label]:.2%}")
        for i, correct_ex in enumerate(correct_examples[true_label][:3]):
            print(f"  Acierto #{i+1} - {class_names[true_label]}: {correct_ex['probs'][true_label]:.2%}")
        print()
else:
    print("No se encontraron errores para visualizar")


In [ ]:
# Análisis detallado: Comparar mapas de calor promedio entre error y aciertos
if len(errors) > 0 and len(correct_examples[errors[0]['label']]) >= 3:
    error = errors[0]
    true_label = error['label']
    pred_label = error['pred']
    
    # Generar CAMs para el error
    error_image_tensor = error['image'].unsqueeze(0).to(device)
    error_cam_true = gradcam.generate_cam(error_image_tensor, class_idx=true_label)
    error_cam_pred = gradcam.generate_cam(error_image_tensor, class_idx=pred_label)
    
    # Generar CAMs para los 3 aciertos
    correct_cams = []
    for correct_example in correct_examples[true_label][:3]:
        correct_image_tensor = correct_example['image'].unsqueeze(0).to(device)
        correct_cam = gradcam.generate_cam(correct_image_tensor, class_idx=true_label)
        correct_cams.append(correct_cam)
    
    # Calcular mapa de calor promedio de los aciertos
    correct_cam_avg = np.mean(correct_cams, axis=0)
    
    # Calcular diferencia entre error y promedio de aciertos
    cam_diff = error_cam_true - correct_cam_avg
    
    # Visualización comparativa
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    
    # Desnormalizar imágenes
    error_img_denorm = denormalize_image(error['image'])
    error_img_np = (error_img_denorm.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    
    # === FILA 1: ERROR ===
    axes[0, 0].imshow(error_img_np, cmap='gray')
    axes[0, 0].set_title('ERROR\nImagen Original', fontsize=12, color='red', fontweight='bold')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(error_cam_true, cmap='jet')
    axes[0, 1].set_title(f'Grad-CAM Error\n({class_names[true_label]})\nMax: {error_cam_true.max():.3f}\nMean: {error_cam_true.mean():.3f}', 
                        fontsize=11, fontweight='bold')
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(error_cam_pred, cmap='jet')
    axes[0, 2].set_title(f'Grad-CAM Error\n({class_names[pred_label]})\nMax: {error_cam_pred.max():.3f}\nMean: {error_cam_pred.mean():.3f}', 
                        fontsize=11, fontweight='bold')
    axes[0, 2].axis('off')
    
    overlaid_error, _ = gradcam.overlay_heatmap(error_img_np, error_cam_true)
    axes[0, 3].imshow(overlaid_error)
    axes[0, 3].set_title('Heatmap Superpuesto\n(Error)', fontsize=11)
    axes[0, 3].axis('off')
    
    # === FILA 2: PROMEDIO DE ACIERTOS ===
    # Usar la primera imagen correcta como referencia visual
    correct_img_denorm = denormalize_image(correct_examples[true_label][0]['image'])
    correct_img_np = (correct_img_denorm.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    
    axes[1, 0].imshow(correct_img_np, cmap='gray')
    axes[1, 0].set_title('ACIERTOS\nImagen de referencia', fontsize=12, color='green', fontweight='bold')
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(correct_cam_avg, cmap='jet')
    axes[1, 1].set_title(f'Grad-CAM Promedio Aciertos\n({class_names[true_label]})\nMax: {correct_cam_avg.max():.3f}\nMean: {correct_cam_avg.mean():.3f}', 
                         fontsize=11, fontweight='bold')
    axes[1, 1].axis('off')
    
    # Diferencia
    axes[1, 2].imshow(cam_diff, cmap='RdBu_r', vmin=-cam_diff.max(), vmax=cam_diff.max())
    axes[1, 2].set_title(f'Diferencia\n(Error - Promedio Aciertos)\nMax: {cam_diff.max():.3f}\nMin: {cam_diff.min():.3f}', 
                         fontsize=11, fontweight='bold')
    axes[1, 2].axis('off')
    
    overlaid_correct, _ = gradcam.overlay_heatmap(correct_img_np, correct_cam_avg)
    axes[1, 3].imshow(overlaid_correct)
    axes[1, 3].set_title('Heatmap Superpuesto\n(Promedio Aciertos)', fontsize=11)
    axes[1, 3].axis('off')
    
    plt.suptitle(f'Análisis Detallado: {class_names[true_label]} clasificado como {class_names[pred_label]}', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Análisis estadístico detallado
    print(f"\n{'='*70}")
    print(f"Análisis Comparativo Detallado")
    print(f"{'='*70}")
    print(f"Error: {class_names[true_label]} → {class_names[pred_label]}")
    print(f"\nEstadísticas de Activación (Clase Real {class_names[true_label]}):")
    print(f"  Error:")
    print(f"    - Máxima: {error_cam_true.max():.3f}")
    print(f"    - Promedio: {error_cam_true.mean():.3f}")
    print(f"    - Desviación estándar: {error_cam_true.std():.3f}")
    print(f"\n  Promedio de 3 Aciertos:")
    print(f"    - Máxima: {correct_cam_avg.max():.3f}")
    print(f"    - Promedio: {correct_cam_avg.mean():.3f}")
    print(f"    - Desviación estándar: {correct_cam_avg.std():.3f}")
    print(f"\n  Diferencia (Error - Aciertos):")
    print(f"    - Máxima: {cam_diff.max():.3f}")
    print(f"    - Mínima: {cam_diff.min():.3f}")
    print(f"    - Promedio: {cam_diff.mean():.3f}")
    print(f"\nProbabilidades:")
    print(f"  Error - {class_names[true_label]}: {error['probs'][true_label]:.2%}")
    print(f"  Error - {class_names[pred_label]}: {error['probs'][pred_label]:.2%}")
    for i, correct_ex in enumerate(correct_examples[true_label][:3]):
        print(f"  Acierto #{i+1} - {class_names[true_label]}: {correct_ex['probs'][true_label]:.2%}")
    
    # Interpretación
    if error_cam_true.mean() < correct_cam_avg.mean():
        print(f"\nINTERPRETACIÓN: El error tiene menor activación promedio que los aciertos.")
        print(f"   Esto sugiere que el modelo no se enfocó lo suficiente en las características")
        print(f"   relevantes de {class_names[true_label]}.")
    else:
        print(f"\nINTERPRETACIÓN: El error tiene mayor activación promedio que los aciertos.")
        print(f"   Esto sugiere que el modelo se enfocó en regiones diferentes o incorrectas.")
else:
    print("No hay suficientes datos para el análisis detallado")


## Conclusiones


*Nota: Ejecutar el notebook y analizar los resultados para completar las conclusiones basadas en los mapas de calor generados para DenseNet121.*
